<a href="https://colab.research.google.com/github/dandreryanboston-oss/DESARROLLO-DE-APLICACIONES-MOVILES/blob/main/VISION_X_AI_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1 — Install the Ultralytics package.
# Keep Colab's existing PyTorch installation for the initial GPU test.

%pip install -q ultralytics

In [ ]:
# cell 2
import torch
from pathlib import Path

PROJECT_DIR = Path("/content/VISION_X_AI")

for folder in ("inputs", "outputs", "evidence"):
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Selected device:", DEVICE)
if DEVICE == "cpu":
    print("Using CPU. YOLO11x image processing will be slower.")
else:
    print("GPU:", torch.cuda.get_device_name(0))

Selected device: cpu
Using CPU. YOLO11x image processing will be slower.


In [ ]:
# cell 3
from ultralytics import YOLO
import torch

DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Loading YOLO11x...", flush=True)
model = YOLO("yolo11x.pt")

print("Model loaded.")
print("Device:", DEVICE)
print("Supported object categories:", len(model.names))

if DEVICE == "cpu":
    print("CPU mode: image processing may take longer.")
else:
    print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Loading YOLO11x...
Model loaded.
Device: cpu
Supported object categories: 80
CPU mode: image processing may take longer.


In [ ]:
#cell 4
from pathlib import Path
import json
import cv2
import numpy as np
import torch

from google.colab import files
from IPython.display import display
from PIL import Image

if "model" not in globals():
    raise RuntimeError("Run Cell 3 first to load YOLO11x.")

print("Choose ONE JPG or PNG image below.", flush=True)
uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError(
        "Select exactly one image. Rerun this cell to try again."
    )

filename, image_bytes = next(iter(uploaded.items()))

# Decode the uploaded image directly from its bytes.
image_array = np.frombuffer(image_bytes, dtype=np.uint8)
frame = cv2.imdecode(image_array, cv2.IMREAD_COLOR)

if frame is None:
    raise ValueError(
        "The file could not be read. Upload a valid JPG or PNG."
    )

output_dir = Path("/content/VISION_X_AI/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

device = 0 if torch.cuda.is_available() else "cpu"

print(
    f"Image loaded: {frame.shape[1]} × {frame.shape[0]} pixels",
    flush=True,
)
print(f"Starting detection on {device}...", flush=True)

if device == "cpu":
    print("Please wait. YOLO11x is slower on CPU.", flush=True)

with torch.inference_mode():
    result = model.predict(
        source=frame,
        device=device,
        imgsz=640,
        conf=0.35,
        verbose=True,
    )[0]

print("Detection finished.", flush=True)

# Draw and save the annotated image.
annotated_bgr = result.plot()
output_path = output_dir / "phase1_annotated.jpg"

if not cv2.imwrite(str(output_path), annotated_bgr):
    raise IOError("Could not save the annotated image.")

annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
display(Image.fromarray(annotated_rgb))

# Display and save structured detection results.
detections = []

for box in result.boxes:
    class_id = int(box.cls.item())
    category = model.names[class_id]
    confidence = float(box.conf.item())

    detections.append({
        "category": category,
        "confidence": confidence,
        "bbox_xyxy": box.xyxy[0].cpu().tolist(),
    })

    print(f"- {category}: {confidence:.1%}")

json_path = output_dir / "phase1_detections.json"
json_path.write_text(
    json.dumps(detections, indent=2),
    encoding="utf-8",
)

print(f"\nCompleted: {len(detections)} detections.")

if not detections:
    print(
        "No objects passed the confidence threshold. "
        "Try a clear image of people, vehicles or common objects."
    )

print("Annotated image saved:", output_path)
print("Detection data saved:", json_path)

Choose ONE JPG or PNG image below.


MessageError: RangeError: Maximum call stack size exceeded.

In [ ]:
# CELL 5 — Upload and check one video.

from google.colab import files
from pathlib import Path
import cv2

print("Select ONE short MP4 video.", flush=True)
uploaded_video = files.upload()

if len(uploaded_video) != 1:
    raise ValueError("Upload exactly one video.")

filename, video_bytes = next(iter(uploaded_video.items()))

video_dir = Path("/content/VISION_X_AI/inputs")
video_dir.mkdir(parents=True, exist_ok=True)

video_path = video_dir / Path(filename).name
video_path.write_bytes(video_bytes)

capture = cv2.VideoCapture(str(video_path))

try:
    if not capture.isOpened():
        raise ValueError("Cannot open this video. Try an MP4 file.")

    fps = capture.get(cv2.CAP_PROP_FPS)
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))

    readable, first_frame = capture.read()
    if not readable:
        raise ValueError("The video opened, but its frames cannot be read.")

    print("\nVideo uploaded and first frame read successfully.")
    print("File:", video_path.name)
    print(f"Resolution: {width} × {height}")
    print(f"Reported FPS: {fps:.2f}")
    print("Reported frame count:", frame_count)

    if fps > 0 and frame_count > 0:
        print(f"Approximate duration: {frame_count / fps:.1f} seconds")

finally:
    capture.release()

Select ONE short MP4 video.


Saving video test multiple.mp4 to video test multiple.mp4

Video uploaded and first frame read successfully.
File: video test multiple.mp4
Resolution: 464 × 832
Reported FPS: 59.97
Reported frame count: 579
Approximate duration: 9.7 seconds


In [ ]:
# CELL 6 — YOLO11x + ByteTrack video test.

from pathlib import Path
import csv
import time
import cv2
import torch
from ultralytics import YOLO

if "video_path" not in globals() or not Path(video_path).exists():
    raise RuntimeError("Run Cell 5 and upload your video first.")

output_dir = Path("/content/VISION_X_AI/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

tracked_video_path = output_dir / "test_tracked.mp4"
tracking_csv_path = output_dir / "test_tracking.csv"

device = 0 if torch.cuda.is_available() else "cpu"

# A fresh model gives this video its own tracking session.
tracking_model = YOLO("yolo11x.pt")

capture = cv2.VideoCapture(str(video_path))
writer = None

processed_frames = 0
unique_ids = set()
unique_person_ids = set()
peak_people = 0
started = time.perf_counter()

try:
    if not capture.isOpened():
        raise RuntimeError("Cannot open the uploaded video.")

    fps = capture.get(cv2.CAP_PROP_FPS)
    total_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps <= 0:
        raise RuntimeError("Invalid video FPS. Try another MP4.")

    ok, frame = capture.read()
    if not ok:
        raise RuntimeError("Cannot read the first video frame.")

    height, width = frame.shape[:2]

    writer = cv2.VideoWriter(
        str(tracked_video_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height),
    )

    if not writer.isOpened():
        raise RuntimeError("Cannot create the output video.")

    print("Device:", device, flush=True)
    print("Processing every frame; no frames intentionally skipped.")
    print("Starting YOLO11x + ByteTrack...", flush=True)

    with tracking_csv_path.open(
        "w", newline="", encoding="utf-8"
    ) as csv_file:
        csv_writer = csv.writer(csv_file)
        csv_writer.writerow([
            "frame_index",
            "video_time_seconds",
            "track_id",
            "category",
            "confidence",
            "x1", "y1", "x2", "y2",
        ])

        while ok:
            with torch.inference_mode():
                result = tracking_model.track(
                    source=frame,
                    persist=True,
                    tracker="bytetrack.yaml",
                    device=device,
                    imgsz=640,
                    conf=0.10,
                    verbose=False,
                )[0]

            people_now = 0

            for box in result.boxes:
                class_id = int(box.cls.item())
                category = tracking_model.names[class_id]
                confidence = float(box.conf.item())
                coordinates = box.xyxy[0].cpu().tolist()

                track_id = (
                    int(box.id.item())
                    if box.id is not None
                    else None
                )

                if category == "person":
                    people_now += 1

                if track_id is not None:
                    unique_ids.add(track_id)
                    if category == "person":
                        unique_person_ids.add(track_id)

                csv_writer.writerow([
                    processed_frames,
                    round(processed_frames / fps, 4),
                    track_id if track_id is not None else "",
                    category,
                    round(confidence, 4),
                    *[round(value, 2) for value in coordinates],
                ])

            peak_people = max(peak_people, people_now)

            annotated = result.plot()

            # Dark banner with readable monitoring statistics.
            cv2.rectangle(
                annotated, (0, 0), (width, 70), (15, 20, 35), -1
            )
            cv2.putText(
                annotated,
                f"VISION X AI | People visible: {people_now}",
                (15, 27),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (255, 255, 255),
                2,
            )
            cv2.putText(
                annotated,
                f"Peak: {peak_people} | Person track IDs: "
                f"{len(unique_person_ids)}",
                (15, 55),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.60,
                (255, 200, 80),
                2,
            )

            writer.write(annotated)
            processed_frames += 1

            if processed_frames == 1 or processed_frames % 10 == 0:
                elapsed = time.perf_counter() - started
                print(
                    f"Processed {processed_frames}/{total_frames} "
                    f"frames | Elapsed: {elapsed:.0f}s",
                    flush=True,
                )

            ok, frame = capture.read()

finally:
    capture.release()
    if writer is not None:
        writer.release()

elapsed_seconds = time.perf_counter() - started

if processed_frames == 0:
    raise RuntimeError("No frames were processed.")

print("\nPROCESSING FINISHED")
print("Frames processed:", processed_frames)
print("Reported input frames:", total_frames)
print("Unique object track IDs:", len(unique_ids))
print("Unique person track IDs:", len(unique_person_ids))
print("Peak visible people:", peak_people)
print(f"Elapsed processing time: {elapsed_seconds:.1f} seconds")
print(
    "Average pipeline processing FPS:",
    f"{processed_frames / elapsed_seconds:.2f}",
)
print("Video saved:", tracked_video_path)
print("Tracking data saved:", tracking_csv_path)

if total_frames > 0 and processed_frames != total_frames:
    print(
        "WARNING: Processed frame count differs from the video's "
        "reported count. Check for an early decoding failure."
    )

Device: 0
Processing every frame; no frames intentionally skipped.
Starting YOLO11x + ByteTrack...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 2 packages in 219ms
Prepared 1 package in 68ms
Installed 1 package in 1ms
 + lap==0.5.14

requirements: AutoUpdate success ✅ 0.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Processed 1/579 frames | Elapsed: 1s
Processed 10/579 frames | Elapsed: 2s
Processed 20/579 frames | Elapsed: 2s
Processed 30/579 frames | Elapsed: 3s
Processed 40/579 frames | Elapsed: 3s
Processed 50/579 frames | Elapsed: 4s
Processed 60/579 frames | Elapsed: 4s
Processed 70/579 frames | Elapsed: 5s
Processed 80/579 frames | Elapsed: 5s
Processed 90/579 frames | Elapsed: 6s
Processed 100/579 frames | Elapsed: 6s
Processed 110/579 frames | Elapsed: 7s
Processed 120/579 frames | Elapsed: 7s
Processed 130/579 frames | Elapsed: 7s
Processed

In [ ]:
#cell 7
from google.colab import files

files.download(str(tracked_video_path))
files.download(str(tracking_csv_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>